# Clustering
This series of notebooks helps data scientists to forecast multiple time series by building models based on the time-series profiling, i.e identifying similar consumption profiles and building a specific model for each cluster. 

Clustering profile of time series data helps in defining the best fitting model by understanding in terms of choice of regressors (calendar variables or temperatures), forecasting algorithm (ARIMA vs Exponential smoothing) and train set (one year or just few days of data). 

The steps to clustering time series are the following:
1. Prepare data by understand the structure of the data (cross-sectional vs panel data), build a full time sequence and treating missing values
2. Indentifying intermittent time series vs smooth time series, by following an **enhanced** classification based on the Syntetos–Boylan–Croston (SBC) use of Coefficient of Variation (CV^2), Average Demand Interval (ADI) and Standard Deviation Demand Interval (SDDI) indicators
   1. Smooth, if ADI<1.32, CV^2<0.49
   2. Intermittent, if ADI>=1.32, CV^2<0.49
   3. Erratic, if ADI<1.32, CV^2>=0.49
   4. Lumpy, if ADI>=1.32, CV^2>=0.49
   5. Unforecastable time, if SDDI >= SDDI threshold, CV^2<0.49
   6. Unforecastable quantity, if SDDI >= SDDI threshold, CV^2>=0.49
3. Clustering with K-Means those time-series identified as **smooth**
   
In this notebook you will follow step 3 on clustering regular or smooth time series with k-means.


# Summary of tasks

Your task in this notebook is to identify consumption patterns that are similar to each other in order to assign the optimal model to forecast their demand​. 

After you have identified the series that is classified as "intermittent" with respect to those "regular or smooth"​, now your **goal** is to perform a k-means cluster analysis only on time series classified as regular or smooth. 

The **expected output** is to label each time series with a cluster number using k-means.

# Implementation

## Packages
Load the packages required for this notebook

In [0]:
# data elaboration functions
import pandas as pd
import numpy as np
from pathlib import Path
import os

# datetime functions
import datetime as dt

# plot functions
import matplotlib.pyplot as plt
import plotly.graph_objects as go

# data science functions
from kneed import KneeLocator
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# statistical functions
from scipy.stats.mstats import winsorize

## Setup & Configuration
Define configuration parameters inline (migrated from config.yaml)

In [0]:
# Time series configuration
date_var = 'WEEK_START_DT'
date_format = '%Y-%m-%d'
id = 'product_name'
unique_id = 'STORE_LOCATION_ID'
frequency = 'W'
y = 'TOTAL_NET_SALES'

# Fabric Lakehouse configuration
LAKEHOUSE_NAME = "ts_forecasting"
INPUT_TABLE = "df_final"
PROFILING_TABLE = "df_profiling"
OUTPUT_TABLE = "df_profiling_clustering"
LOCAL_FOLDER = "supermarket_net_sales"  # Local folder for fallback storage

root_path = Path.cwd().parent.parent

# Winsorizing parameters
highest = 0.05
lowest = 0.05

# Clustering parameters
try_clusters = 5

# Set seed for reproducibility
sample_seed_kmeans = 789

print("Configuration loaded:")
print(f"  - Date variable: {date_var}")
print(f"  - Target variable: {y}")
print(f"  - Unique ID: {unique_id}")
print(f"  - ID: {id}")
print(f"  - Max clusters to try: {try_clusters}")
print(f"  - Winsorizing: highest={highest}, lowest={lowest}")

## Helper Functions
Utility functions for plotting (migrated from Plotting.plots module)

In [0]:
def find_date(df: pd.DataFrame) -> str:
    """
    Finds date columns in a dataframe
    :params: df as dataframe
    :return: a string
    """
    dates = list(df.select_dtypes(include=['datetime','datetime64[ns, UTC]', 'datetime64[ns]']).drop_duplicates().columns)
    
    if len(dates) == 1:
        date_col = dates[0]
    elif len(dates) == 0:
        dates = list(df.select_dtypes(include=['period[M]']).drop_duplicates().columns)
        date_col = dates[0] if dates else None
    else:
        date_col = dates[0]
    
    return str(date_col)







## Load Data
Load data from Fabric Lakehouse tables

In [0]:
# Read from Lakehouse table using Spark, then convert to Pandas
try:
    df_spark = spark.table(f"{LAKEHOUSE_NAME}.{INPUT_TABLE}")
    df_final = df_spark.toPandas()
except:
    print(f"❌ Error loading data from {LAKEHOUSE_NAME}.{INPUT_TABLE}, loading from local file instead.")
    root_path = Path.cwd().parent.parent
    df_final = pd.read_parquet(f"{root_path}/data/{LOCAL_FOLDER}/{INPUT_TABLE}.parquet")  # Fallback to local file if Lakehouse load fails

print(f"✅ Loaded {len(df_final)} rows")
print(df_final.head())

In [0]:
try:
    df_spark = spark.table(f"{LAKEHOUSE_NAME}.{PROFILING_TABLE}")
    df_profiling = df_spark.toPandas()
except:
    print(f"❌ Error loading data from {LAKEHOUSE_NAME}.{PROFILING_TABLE}, loading from local file instead.")
    root_path = Path.cwd().parent.parent
    df_profiling = pd.read_parquet(f"{root_path}/data/{LOCAL_FOLDER}/{PROFILING_TABLE}.parquet")  # Fallback to local file if Lakehouse load fails

print(f"✅ Loaded {len(df_profiling)} rows")
print(df_profiling.head())

# Clustering regular time series

Define regular ids list

In [0]:
list_id_profiling = list(df_profiling.loc[df_profiling['profile']=='regular', unique_id].unique())
print(f"Number of regular time series: {len(list_id_profiling)}")

mask = df_final[unique_id].isin(list_id_profiling)
df = df_final.loc[mask==True, ].copy()
print(f"Filtered dataframe rows: {len(df)}")
df.head()

## Winsorizing data

In [0]:
df_win_sum = df.groupby([unique_id, date_var])[y].apply(
    lambda x: np.sum(winsorize(x, (highest, lowest)))).reset_index()
df_win_sum.columns = [unique_id, date_var, "sum_" + y]
print(f"Winsorized data shape: {df_win_sum.shape}")
df_win_sum.head()

## Standardizing data
Checking if some ids have 0 values after winsorizing and standardise the target variable

In [0]:
scaler = StandardScaler()

if len(set(list_id_profiling) - set(list(df_win_sum[unique_id].unique()))) > 0:
    list_id_profiling = list(set(list_id_profiling) - set(list(df_win_sum[unique_id].unique())))
    print(unique_id, list_id_profiling, "has/have 0", y, "after winsorizing")
    mask = (df_win_sum[y]!=np.nan) & (~df_win_sum[unique_id].isin(list_id_profiling))
    
    print("Standardizing data for ids without NA values after winsorizing")
    df_win_sum['y_std'] = scaler.fit_transform(df_win_sum[[y]])   
    df_std = df_win_sum.loc[mask, ].pivot(index=date_var, columns=id, values='y_std').reset_index()
    charvec = df_std[date_var].dt.strftime('%Y-%m-%d')
    df_std.set_index(date_var, inplace=True)
else:
    mask = (df[y]!=np.nan)
    
    print("Standardizing data for ids without NA values after winsorizing")
    df['y_std'] = scaler.fit_transform(df[[y]])    
    df_std = df.loc[mask, ].pivot(index=date_var, columns=unique_id, values='y_std').reset_index()
    charvec = df_std[date_var].dt.strftime('%Y-%m-%d')
    df_std.set_index(date_var, inplace=True)
    print("NO", id, "has/have 0", y, "after winsorizing")
    
df_std.head()

## Defining a set of ids to cluster with NO nan
In order to perform cluster analysis, one need to have a matrix with no nan value and set the index of the dataframe with date_var.
The dataframe has to have unique_id as index and the datetime as column names.

In [0]:
df_std_no_nan = df_std.dropna()
print('Number of ids with complete data:', len(df_std_no_nan))

if len(df_std_no_nan) == 0:
    list_id_cluster = []
    df_cluster = df_std.loc[:, list_id_cluster].dropna().T
else:
    list_id_cluster = list(set(list(df_std.columns)) - set([date_var]))
    df_cluster = df_std.loc[:, list_id_cluster].dropna().T
    
print('Clustering regular profiles on ids:', list_id_cluster)
df_cluster.head()

## ✅ CHECK POINT with the data scientist: set the number of clusters to try

In [0]:
# Setting up date range
start_date = min(df_cluster.index) if len(df_cluster) > 0 else None
print('Start date:', start_date)
end_date = max(df_cluster.index) if len(df_cluster) > 0 else None
print('End date:', end_date)

# Define the number of clusters
print("Try clusters:", try_clusters)

# K-means setup
kmeans_kwargs = { 
    "init": "random",
    "n_init": 10,
    "max_iter": 300,
    "random_state": 42,
}

## Choosing the Appropriate Number of Clusters
In this section, you'll look at two methods that are commonly used to evaluate the appropriate number of clusters:

- The elbow method
- The silhouette coefficient

These are often used as complementary evaluation techniques

### The elbow method

In [0]:
X = np.array(df_cluster)

# A list holds the SSE values for each k
sse = []
for k in range(1, try_clusters):
    kmeans = KMeans(n_clusters=k, **kmeans_kwargs)
    kmeans.fit_predict(X)
    sse.append(kmeans.inertia_)

plt.style.use("fivethirtyeight")
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(range(1, try_clusters), sse, marker='o')
ax.set_xticks(range(1, try_clusters))
ax.set_xlabel("Number of Clusters")
ax.set_ylabel("SSE")
ax.set_title("Elbow Method for Optimal k")

chart_name = f"Notebook_04_elbow_method_try_clusters_{try_clusters}"
chart_path = os.path.join(root_path, "data", LOCAL_FOLDER, "charts", chart_name + ".png")
os.makedirs(os.path.dirname(chart_path), exist_ok=True)
fig.savefig(chart_path, dpi=900, bbox_inches='tight')
plt.show()

In [0]:
kl = KneeLocator(range(1, try_clusters), sse, curve="convex", direction="decreasing")
print("Elbow method: optimal number of clusters is", kl.elbow)

### The silhouette coefficient
The silhouette coefficient is a measure of cluster cohesion and separation. It quantifies how well a data point fits into its assigned cluster based on two factors:

- How close the data point is to other points in the cluster
- How far away the data point is from points in other clusters

Silhouette coefficient values range between -1 and 1. Larger numbers indicate that samples are closer to their clusters than they are to other clusters.

In [0]:
# A list holds the silhouette coefficients for each k
silhouette_coefficients = []

# Notice you start at 2 clusters for silhouette coefficient
for k in range(2, try_clusters):
    kmeans = KMeans(n_clusters=k, **kmeans_kwargs)
    kmeans.fit(X)
    score = silhouette_score(X, kmeans.labels_)
    silhouette_coefficients.append(score)
    
print("Silhouette coefficients:")
print(pd.DataFrame(silhouette_coefficients, index=range(2, try_clusters), columns=['Silhouette Score']))
    
plt.style.use("fivethirtyeight")
plt.figure(figsize=(10, 6))
plt.plot(range(2, try_clusters), silhouette_coefficients, marker='o')
plt.xticks(range(2, try_clusters))
plt.xlabel("Number of Clusters")
plt.ylabel("Silhouette Coefficient")
plt.title("Silhouette Method for Optimal k")

chart_name = f"Notebook_04_silhouette_method_try_clusters_{try_clusters}"
chart_path = os.path.join(root_path, "data", LOCAL_FOLDER, "charts", chart_name + ".png")
os.makedirs(os.path.dirname(chart_path), exist_ok=True)
plt.savefig(chart_path, dpi=900, bbox_inches='tight')
plt.show()

df_sil_coeff = pd.DataFrame(silhouette_coefficients).reset_index()
optimal_silhouette_coefficients = df_sil_coeff.loc[df_sil_coeff[0] == max(silhouette_coefficients), 'index'].values[0] + 2
print("Silhouette coefficients: optimal number of clusters is", optimal_silhouette_coefficients)

## ✅ CHECK POINT with the data scientist: cluster using the chosen number of clusters

In [0]:
# Choose the number of clusters based on the analysis above
chosen_clusters = max(kl.elbow, optimal_silhouette_coefficients)
print(f"Chosen number of clusters: {chosen_clusters}")

In [0]:
kmeans = KMeans(n_clusters=chosen_clusters, **kmeans_kwargs)
identified_clusters = kmeans.fit_predict(X)

df_cluster.loc[:, 'cluster'] = identified_clusters 
print(f"Clustering complete. Shape: {df_cluster.shape}")
df_cluster.head()

In [0]:
print("List of IDs for profiling:")
print(list_id_profiling)

In [0]:
dict_cluster = {}
dict_cluster['regular'] = {}
for c in range(0, len(list_id_profiling)):
    if list_id_profiling[c] in df_cluster.index:
        dict_cluster['regular'][c] = int(df_cluster.loc[df_cluster.index == list_id_profiling[c], 'cluster'].unique()[0])

count = 0
for cluster in range(0, chosen_clusters):
    num_cluster = len(df_cluster.loc[df_cluster['cluster'] == cluster].index.tolist())
    count = count + num_cluster
    print("Unique ids in cluster", cluster, 'are', num_cluster)
    
print("\nTotal unique ids clustered:", count)

## Plotting clustered regular series

In [0]:
df_cluster.reset_index().head()

### Reshaping data frame for plots

In [0]:
df_to_plot = pd.melt(df_cluster.reset_index(), id_vars=[unique_id, 'cluster'], var_name=date_var, value_name='y_std')
df_to_plot[date_var] = pd.to_datetime(df_to_plot[date_var])
df_to_plot = pd.merge(df_to_plot, right=df[[unique_id, date_var, y]].drop_duplicates(), on=[unique_id, date_var], how='left', validate='1:1')
print(f"Data for plotting prepared: {len(df_to_plot)} rows")
df_to_plot.head()

In [0]:
print("Num of regular time series clusterized:", df_to_plot[[unique_id, 'cluster']].drop_duplicates().groupby('cluster').count().sum()[0])
df_to_plot[[unique_id, 'cluster']].drop_duplicates().groupby('cluster').count()

### ✅ CHECK POINT with the data scientist: plot max 10 time series per cluster

Plot up to 10 time series per cluster chosen randomly, so that the data scientist can look at them and maybe give you hints on the drivers of consumption, such as weather data or promotional marketing campaigns.

These information are going to be useful for the feature engineering step.

In [0]:
# Replace only the overview-plot section inside your loop
plot_df = (
    df_to_plot.loc[df_to_plot[unique_id].isin(plot_ids), :]
    .groupby([unique_id, date_var])[y]
    .mean()
    .reset_index()
    .pivot(index=date_var, columns=unique_id, values=y)
)

ax = plot_df.plot(figsize=(15, 7))
ax.set_title(f"Demand per product over time, cluster {cluster}")
fig = ax.get_figure()
fig.tight_layout()

chart_name = f"Notebook_04_demand_per_product_cluster_{cluster}"
chart_path = os.path.join(root_path, "data", LOCAL_FOLDER, "charts", chart_name + ".png")
os.makedirs(os.path.dirname(chart_path), exist_ok=True)

fig.savefig(chart_path, dpi=900, bbox_inches="tight")
plt.show()
plt.close(fig)

In [0]:

# Store plots in a dictionary for display
cluster_plots = {}
import random
random.seed(sample_seed_kmeans)

for cluster in sorted(list(df_to_plot['cluster'].unique())):
    print(f'\n=== Plotting cluster: {cluster} ===')
    list_ids_in_cluster = list(df_to_plot.loc[df_to_plot['cluster'] == cluster, unique_id].unique())
    plot_ids = list_ids_in_cluster if len(list_ids_in_cluster) <= 10 else random.sample(list_ids_in_cluster, 10)
    
    # Create overview plot for cluster
    plt.figure(figsize=(15, 7))
    if len(list_ids_in_cluster) > 10:
        print('Plotting 10 randomly chosen ids in cluster for visualization')
    df_to_plot.loc[df_to_plot[unique_id].isin(plot_ids), ].groupby([unique_id, date_var])[y].apply(np.mean).reset_index().pivot(index=date_var, columns=unique_id, values=y).plot(figsize=(15,7))
    fig = ax.get_figure()
    plt.title(f'Demand per product over time, cluster {cluster}')
    plt.tight_layout()
    
    chart_name = f"Notebook_04_demand_per_product_cluster_{cluster}"
    chart_path = os.path.join(root_path, "data", LOCAL_FOLDER, "charts", chart_name + ".png")
    fig.savefig(chart_path, dpi=900, bbox_inches="tight")
    plt.show()
    
    # Create individual interactive plots
    count = 1
    for i in plot_ids:
        print(f'Creating plot for id: {i} ({count} of {len(plot_ids)})')
        
        chart_title = f"{unique_id} {i} - Profile regular cluster {cluster}"
        plot = sliding_line_plot(df_to_plot, unique_id, y, i, chart_title=chart_title)
        
        save_path = os.path.join(root_path, "data", LOCAL_FOLDER, "charts", f"{unique_id}_{i}_cluster_{cluster}.html")
        plot.write_html(save_path)
        
        # Store plot for optional display
        cluster_plots[f"{unique_id}_{i}_cluster_{cluster}"] = plot
        count += 1
        
print(f"\n✅ Total plots created: {len(cluster_plots)}")

In [0]:
# Display a sample plot (first one from each cluster)
for cluster in sorted(list(df_to_plot['cluster'].unique())):
    keys_for_cluster = [k for k in cluster_plots.keys() if k.endswith(f"_cluster_{cluster}")]
    if not keys_for_cluster:
        print(f"\nNo stored plot found for Cluster {cluster}")
        continue
    plot_key = keys_for_cluster[0]
    print(f"\nSample plot for Cluster {cluster}:")
    cluster_plots[plot_key].show()

⚠️ Save also the additional information on potential drivers the data scientist is providing you with.

# Saving
The output is a dataframe with the same information as in df_profiling, with an additional column "cluster" with the cluster number only for regular profiles

In [0]:
df_profiling_clustering = pd.merge(
    df_profiling, 
    df_cluster[['cluster']].reset_index().rename(columns={'index': unique_id}), 
    on=unique_id, 
    how='left', 
    validate='1:1'
)
print(f"Merged profiling with clusters: {len(df_profiling_clustering)} rows")

# Pivot table showing profile vs cluster distribution
pd.pivot_table(df_profiling_clustering, index='profile', columns='cluster', values=unique_id, aggfunc='count', fill_value=0).T
print("Total number of series:", pd.pivot_table(df_profiling_clustering, index='profile', columns='cluster', values=unique_id, aggfunc='count', fill_value=0).T.sum().sum())

df_profiling_clustering['profile_cluster'] = df_profiling_clustering['profile']
df_profiling_clustering.loc[df_profiling_clustering['profile'] == 'regular', 'profile_cluster'] = df_profiling_clustering['cluster']
df_profiling_clustering.drop(columns=['profile', 'cluster'], inplace=True)
df_profiling_clustering['profile_cluster'] = df_profiling_clustering['profile_cluster'].astype(str)
df_profiling_clustering.head()

In [0]:
# Save results to Lakehouse
try:
    df_profiling_clustering_spark = spark.createDataFrame(df_profiling_clustering)
    df_profiling_clustering_spark.write.mode("overwrite").saveAsTable(f"{LAKEHOUSE_NAME}.{OUTPUT_TABLE}")
    print(f"✅ Results saved to Lakehouse table: {LAKEHOUSE_NAME}.{OUTPUT_TABLE}")
except:
    df_profiling_clustering.to_parquet(f"{root_path}/data/{LOCAL_FOLDER}/{OUTPUT_TABLE}.parquet", index=False)
    print(f"❌ Error saving results to Lakehouse table")
    print("Saving to local file instead.")
    print(f"✅ df_features saved locally at: {root_path}/data/{OUTPUT_TABLE}.parquet") 